# 01 — Bronze: Ingestão NYC TLC → S3

**Responsabilidade:** Armazenar os arquivos originais no S3 e registrar no Unity Catalog como tabela Delta preservando dado 100% original.

> Executar `00_config` antes deste notebook.

**Nota sobre download:** O Databricks Serverless não tem acesso à internet externa.
O download dos arquivos é feito localmente via `src/ingestion/download_to_s3.py`.

## Célula 1 — Carregar configurações

In [ ]:
%run "./00_config"

In [ ]:
import requests, io, boto3
from botocore.exceptions import ClientError
from pyspark.sql import functions as F

## Célula 2 — Verificar arquivos no S3

In [ ]:
# Verifica se os arquivos foram carregados corretamente

# Credenciais via Databricks Secrets — nunca expostas na UI
AWS_KEY    = dbutils.secrets.get(scope="ifood-aws", key="access-key")
AWS_SECRET = dbutils.secrets.get(scope="ifood-aws", key="secret-key")

s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
)

def _exists(key):
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] == "404":
            return False
        raise

def ingest_month(month):
    filename = f"yellow_tripdata_{YEAR}-{month}.parquet"
    url      = f"{TLC_BASE_URL}/{filename}"
    s3_key   = f"bronze/nyc_taxi/yellow/year={YEAR}/month={month}/{filename}"
    if _exists(s3_key):
        print(f"  Ja existe: {filename}")
        return {"month": month, "status": "skipped"}
    print(f"  Baixando: {filename}")
    resp = requests.get(url, stream=True, timeout=180)
    resp.raise_for_status()
    content = resp.content
    mb = len(content) / 1_048_576
    print(f"  Subindo {mb:.1f} MB -> S3")
    s3.upload_fileobj(io.BytesIO(content), S3_BUCKET, s3_key)
    print(f"  OK: {filename} ({mb:.1f} MB)")
    return {"month": month, "status": "uploaded", "mb": round(mb, 1)}

print("=== Bronze Ingestion ===")
results = []
for month in MONTHS:
    print(f"\nMes {month}:")
    results.append(ingest_month(month))

print("\n=== Resumo ===")
for r in results:
    icon = "OK" if r["status"] in ("uploaded", "skipped") else "ERRO"
    info = f'{r.get("mb","")} MB' if r["status"] == "uploaded" else r["status"]
    print(f"  [{icon}] Mes {r['month']} -> {info}")

## Célula 3 — Criar schema Bronze e registrar tabela Delta

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
print(f"Schema: {CATALOG}.bronze")

# Lê cada mês separadamente preservando o dado 100% original
# unionByName: alinha colunas pelo nome e preenche NULL onde não existe
# Resolve o problema de airport_fee vs Airport_fee sem modificar nada
dfs = []
for month in MONTHS:
    path = f"s3://{S3_BUCKET}/bronze/nyc_taxi/yellow/year={YEAR}/month={month}/"
    print(f"Lendo mes {month}...")
    df = spark.read.parquet(path) \
           .withColumn("year",  F.lit(YEAR)) \
           .withColumn("month", F.lit(month))
    dfs.append(df)
    print(f"  OK: {len(df.columns)} colunas")

# Une todos os meses mantendo TODAS as colunas originais
df_bronze = dfs[0]
for df in dfs[1:]:
    df_bronze = df_bronze.unionByName(df, allowMissingColumns=True)

print(f"Total colunas: {len(df_bronze.columns)}")
df_bronze.printSchema()

# Salva como Delta no Unity Catalog — dado original intacto
# mode("overwrite") é atômico no Delta Lake — DROP TABLE antes é desnecessário
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"\nTabela registrada: {BRONZE_TABLE}")
print(f"  Tipo: Delta — dado original completo, sem transformações")

## Célula 4 — Validação Bronze

In [ ]:
spark.sql(f"DESCRIBE TABLE {BRONZE_TABLE}").show(30, truncate=False)
spark.sql(f'''
    SELECT year, month, COUNT(*) AS total_registros
    FROM {BRONZE_TABLE}
    GROUP BY year, month ORDER BY year, CAST(month AS INT)
''').show()
spark.sql(f'''
    SELECT VendorID, passenger_count, total_amount,
           tpep_pickup_datetime, tpep_dropoff_datetime
    FROM {BRONZE_TABLE} LIMIT 5
''').show(truncate=False)